In [1]:
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from difflib import SequenceMatcher

In [2]:
# Načtení CSV souborů
t1 = pd.read_csv("../invalid/data/T1.csv")
raw = pd.read_csv("../invalid/data/RAW_CONTACTS.csv")

# Výběr relevantních sloupců
t1_subset = t1[["PARTY_ID", "PARTY_LNAME_STD", "PARTY_FNAME_STD", "CONT_EMAIL"]]
raw_subset = raw[["PARTY_ID", "CONT_VALUE"]]

# Sloučení obou sad podle PARTY_ID
merged = pd.merge(t1_subset, raw_subset, on="PARTY_ID")
merged = merged.rename(columns={"CONT_EMAIL": "CLEAN_EMAIL", "CONT_VALUE": "RAW_EMAIL"})

In [3]:
# Odstranění záznamů bez obou e-mailů
merged = merged.dropna(subset=["RAW_EMAIL", "CLEAN_EMAIL"])
merged.head(10)

,PARTY_ID,PARTY_LNAME_STD,PARTY_FNAME_STD,CLEAN_EMAIL,RAW_EMAIL
0,1,Kováčová,Erdenetuya,erdenetuya.kovicovi@seznam.cz,erdenetuya.kovicovi@seznam.cz
1,2,Němec,Valter,vnemec@volny.cz,vnemec@volny.cz
2,3,Strnadova,Elena,elena@strnadovi.com,elena@strnadovi.com
3,4,Šulcová,Hasmik,hasmik@sulcovi.eu,hasmik@sulcovi.eu
4,5,Holubová,Luisa,lholubovi@gmail.com,lholubovi#gmail&com
5,6,Kučerová,Isabel,ikucerovi@gmail.cz,ikucerovi@gmail.cz
6,7,Čechova,Michele,michele_cechovi@volny.cz,michele_cechovi@volny.cz
7,8,Šebestová,Thanh,tsebestovi@gmail.cz,tsebestovi@gmail.cz
8,10,Kuchařová,Oksana,oksanaivanivna_kucharovi@volny.cz,oksana ivanivna_kucharovi@volny.cz
9,11,Mrázová,Thanh,thanhhai_mrizovi@volny.cz,thanh hai_mrizovi@volny.cz


In [4]:
# Rozdělení na trénovací a testovací sadu
train_df, test_df = train_test_split(merged, test_size=0.2, random_state=42)

In [5]:
# Tokenizace podle typu znaku (alnum vs special)
def tokenize_by_type(text):
    text = str(text).strip().lower()
    tokens = []
    current = ''
    prev_type = None
    for char in text:
        curr_type = 'alnum' if char.isalnum() or char == '_' else 'special'
        if curr_type != prev_type and current:
            tokens.append(current)
            current = ''
        current += char
        prev_type = curr_type
    if current:
        tokens.append(current)
    return tokens

In [6]:
# Rozdělení e-mailu na lokální část, SLD a TLD
def split_email_parts(email):
    email = str(email).strip().lower()
    if '@' in email:
        local_part, domain_part = email.split('@', 1)
    else:
        local_part, domain_part = email, ''
    domain_tokens = domain_part.split('.') if domain_part else []
    sld = domain_tokens[0] if len(domain_tokens) > 0 else ''
    tld = domain_tokens[1] if len(domain_tokens) > 1 else ''
    return tokenize_by_type(local_part), sld, tld

# Sestavení kompletní sady tokenů z e-mailu
def tokenize_extended(email):
    local_tokens, sld, tld = split_email_parts(email)
    tokens = local_tokens
    if sld:
        tokens += ['@', sld]
    if tld:
        tokens.append(tld)
    return tokens

In [7]:
# Zarovnání tokenů pomocí SequenceMatcher
def diff_aligned(raw_tokens, clean_tokens):
    matcher = SequenceMatcher(None, raw_tokens, clean_tokens)
    diffs = []
    for opcode, i1, i2, j1, j2 in matcher.get_opcodes():
        if opcode == "replace":
            for r, c in zip(raw_tokens[i1:i2], clean_tokens[j1:j2]):
                diffs.append(f"{r}→{c}")
        elif opcode == "delete":
            for r in raw_tokens[i1:i2]:
                diffs.append(f"{r}→[missing]")
        elif opcode == "insert":
            for c in clean_tokens[j1:j2]:
                diffs.append(f"[missing]→{c}")
    return diffs

In [8]:
# Aplikace na trénovací data
train_df["RAW_TOKENS"] = train_df["RAW_EMAIL"].apply(tokenize_extended)
train_df["CLEAN_TOKENS"] = train_df["CLEAN_EMAIL"].apply(tokenize_extended)
train_df["DIFF_TOKENS"] = train_df.apply(
    lambda row: diff_aligned(row["RAW_TOKENS"], row["CLEAN_TOKENS"]), axis=1)

In [9]:
# Vygenerování pravidel
train_transactions = train_df["DIFF_TOKENS"].tolist()
all_diffs = [token for diff_list in train_transactions for token in diff_list]

# Filtrace jednoduchých pravidel (krátké změny, bez [missing])
filtered_diffs = [
    t for t in all_diffs
    if "→" in t and "[missing]" not in t
    and len(t.split("→")[0]) <= 2 and len(t.split("→")[1]) <= 2]

In [10]:
# Výpočet četnosti a uložení do JSON
token_counts = Counter(filtered_diffs)
rules_df = pd.DataFrame(token_counts.items(), columns=["token", "count"])
rules_df = rules_df.sort_values(by="count", ascending=False)
rules_df.to_json("../invalid/export/email_rules.json", orient="records", indent=2, force_ascii=False)

In [11]:
# Vyhodnocení pokrytí pravidel na testovací sadě
test_df["RAW_TOKENS"] = test_df["RAW_EMAIL"].apply(tokenize_extended)
test_df["CLEAN_TOKENS"] = test_df["CLEAN_EMAIL"].apply(tokenize_extended)
test_df["DIFF_TOKENS"] = test_df.apply(
    lambda row: diff_aligned(row["RAW_TOKENS"], row["CLEAN_TOKENS"]), axis=1)

In [12]:
learned_rules = set(rules_df["token"])
test_df["COVERED"] = test_df["DIFF_TOKENS"].apply(
    lambda diff: any(d in learned_rules for d in diff))

In [13]:
# Výpočet matched pravidel pro každý záznam
def match_rules(diff_tokens, rules_set):
    return [d for d in diff_tokens if d in rules_set]

# Přidej sloupec s nalezenými pravidly
test_df["MATCHED_RULES"] = test_df["DIFF_TOKENS"].apply(
    lambda diffs: match_rules(diffs, learned_rules))

# Filtrace záznamů, kde MATCHED_RULES není prázdné
filtered_df = test_df[test_df["MATCHED_RULES"].apply(lambda x: len(x) > 0)]

# Výběr relevantních sloupců
result_json = filtered_df[[
    "PARTY_ID", "RAW_EMAIL", "CLEAN_EMAIL", "DIFF_TOKENS", "MATCHED_RULES", "COVERED"
]].to_dict(orient="records")

# Uložení do souboru
import json
with open("../invalid/export/email_results.json", "w", encoding="utf-8") as f:
    json.dump(result_json, f, indent=2, ensure_ascii=False)